# AnimationStudio — Pre-flight validation (one clear image per step)

Run this notebook **before** the full generation notebook. It walks each stage
of the stack and produces a single clear test image so you can confirm the
pipeline works end to end — without burning hours on a full Phase-1 run.

Steps:

1. GPU / VRAM available + studio installed
2. ComfyUI installed
3. Flux model downloaded to the Colab disk
4. ComfyUI server started and fully answering the API
5. The model is visible to ComfyUI (right filename, right loader)
6. The workflow is **Flux-correct** (cfg=1.0, scheduler=simple — SD-style
   CFG values are why Flux output comes out blurred/over-saturated)
7. **One** 1024×1024 test image is generated, saved to the repo, and shown
   inline
8. Automatic sharpness check on that image

**If you see `Connection refused` or `NameError` at any step:** the Colab kernel/
VM restarted and wiped cell state. Re-run Cells 0–3, then the failing cell —
steps 4–7 auto-restart the server and import the shared helper themselves, so
you do not need to run them in strict order.

If step 8 shows a sharp, visible image → proceed to the main notebook.
If it shows a flat/blurry image → stop; the troubleshooting list at the end
applies.

In [ ]:
#@title 0. Settings

import os
import subprocess
import sys
import time
import shutil
from pathlib import Path

REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}
BRANCH = "colab-gpu"  #@param ["colab-gpu", "master"]

COMFYUI_PORT = 8188  #@param {type:"integer"}
TEST_SIZE = 1024  #@param {type:"integer"}
TEST_SEED = 42  #@param {type:"integer"}
TEST_PROMPT = "Lily Bunny, cute anthropomorphic white rabbit child, fluffy fur, big round expressive eyes, soft studio lighting, crisp clean 3D render, bright cheerful colors, high detail, sharp focus"  #@param {type:"string"}
TEST_NEGATIVE = "blurry, out of focus, low quality, deformed, distorted, text, watermark, logo"  #@param {type:"string"}

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"
COMFY = f"{WORK}/comfyui"

if REPO_URL.startswith("https://github.com/YOUR_ORG/"):
    raise SystemExit("Set REPO_URL in Cell 0 before running.")


def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)

In [ ]:
#@title 1. STEP 1 — Clone repo, install studio, check GPU/VRAM

os.chdir(WORK)
if not os.path.isdir(REPO):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q", "requests", "pillow", "numpy", "IPython"])

import torch
assert torch.cuda.is_available(), "CUDA not available on this runtime"
print("PASS: GPU =", torch.cuda.get_device_name(0))
print("      VRAM =", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
#@title 2. STEP 2 — Install ComfyUI (and ComfyUI-GGUF on master)

if not os.path.isdir(COMFY):
    run(["git", "clone", "--depth", "1",
         "https://github.com/comfyanonymous/ComfyUI.git", COMFY])
run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY}/requirements.txt"])

if BRANCH == "master":
    gguf = f"{COMFY}/custom_nodes/ComfyUI-GGUF"
    if not os.path.isdir(gguf):
        run(["git", "clone", "--depth", "1",
             "https://github.com/city96/ComfyUI-GGUF.git", gguf])
    run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{gguf}/requirements.txt"])
print("PASS: ComfyUI installed at", COMFY)

In [ ]:
#@title 4. STEP 4 — Start ComfyUI (via the shared helper)

# ensure_comfyui_up() is imported from colab/comfy_helpers.py so later cells
# can auto-restart the server after a Colab VM recycle instead of failing
# with Connection refused.  It also survives kernel restarts.

REPO = globals().get("REPO", "/content/AnimationStudio")
COMFY = globals().get("COMFY", "/content/comfyui")
WORK = globals().get("WORK", "/content")
COMFYUI_PORT = globals().get("COMFYUI_PORT", 8188)

import sys
sys.path.insert(0, f"{REPO}/colab")
from comfy_helpers import ensure_comfyui_up, comfy_alive, server_url  # noqa

ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)


In [ ]:
#@title 5. STEP 5 — Verify the model is visible to ComfyUI

# This cell is self-healing: it starts the server itself if STEP 4 did not
# run in this kernel (e.g. after a VM restart).
REPO = globals().get("REPO", "/content/AnimationStudio")
COMFY = globals().get("COMFY", "/content/comfyui")
WORK = globals().get("WORK", "/content")
COMFYUI_PORT = globals().get("COMFYUI_PORT", 8188)
MODEL_FILE = globals().get("MODEL_FILE", None)

import sys
import requests
sys.path.insert(0, f"{REPO}/colab")
from comfy_helpers import ensure_comfyui_up

ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)

assert MODEL_FILE, "Model not downloaded — re-run Cells 0-3 first (or STEP 3 after a VM restart)."

info = requests.get(f"http://127.0.0.1:{COMFYUI_PORT}/object_info", timeout=60).json()


def options(node, key):
    try:
        return sorted(list(info[node]["input"]["required"][key][0]))
    except Exception:
        return []


checkpoints = options("CheckpointLoaderSimple", "ckpt_name")
unets = options("UnetLoaderGGUF", "unet_name")
clips = options("DualCLIPLoader", "clip_name1")

print("checkpoints:", checkpoints)
print("gguf unets: ", unets)
print("dual clips: ", clips)

visible = MODEL_FILE in checkpoints or MODEL_FILE in unets
print("PASS: model visible to ComfyUI" if visible else
      "FAIL: model file not found by ComfyUI — check STEP 3 symlinks")
assert visible, f"Expected to see {MODEL_FILE} in the loader lists above"


In [ ]:
#@title 5. STEP 5 — Verify the model is visible to ComfyUI

ensure_comfyui_up()

def _options(node, key):
    try:
        info = requests.get(f"http://127.0.0.1:{COMFYUI_PORT}/object_info", timeout=60).json()
        return sorted(list(info[node]["input"]["required"][key][0]))
    except Exception:
        return []

checkpoints = _options("CheckpointLoaderSimple", "ckpt_name")
unets = _options("UnetLoaderGGUF", "unet_name")
clips = _options("DualCLIPLoader", "clip_name1")

print("checkpoints:", checkpoints)
print("gguf unets: ", unets)
print("dual clips: ", clips)

visible = MODEL_FILE in checkpoints or MODEL_FILE in unets
print("PASS: model visible to ComfyUI" if visible else
      "FAIL: model file not found by ComfyUI — check Cell 3 symlinks")
assert visible, f"Expected to see {MODEL_FILE} in the loader lists above"

In [ ]:
#@title 7. STEP 7 — Generate ONE clear test image (via ComfyUI)

# Self-healing: starts the server itself if it is not running.
REPO = globals().get("REPO", "/content/AnimationStudio")
COMFY = globals().get("COMFY", "/content/comfyui")
WORK = globals().get("WORK", "/content")
COMFYUI_PORT = globals().get("COMFYUI_PORT", 8188)

import sys
sys.path.insert(0, f"{REPO}/colab")
from comfy_helpers import ensure_comfyui_up

ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)

out = backend.generate(gen_input, asset_type="")
assert out.images, f"No image returned: {out.metadata}"

img = out.images[0]
dst_dir = Path(REPO) / "Universe" / "_validate"
dst_dir.mkdir(parents=True, exist_ok=True)
dst = dst_dir / f"smoke_{TEST_SEED}.png"
img.save(dst, format="PNG")
print("saved:", dst)
print("size :", img.size, "|", f"{dst.stat().st_size / 1024:.1f} KB")

from IPython.display import Image as IPImage, display
display(IPImage(filename=str(dst)))
print("PASS: image generated and saved")


In [ ]:
#@title 7. STEP 7 — Generate ONE clear test image (via ComfyUI)

ensure_comfyui_up()

out = backend.generate(gen_input, asset_type="")
assert out.images, f"No image returned: {out.metadata}"

img = out.images[0]
dst_dir = Path(REPO) / "Universe" / "_validate"
dst_dir.mkdir(parents=True, exist_ok=True)
dst = dst_dir / f"smoke_{TEST_SEED}.png"
img.save(dst, format="PNG")
print("saved:", dst)
print("size :", img.size, "|", f"{dst.stat().st_size / 1024:.1f} KB")

from IPython.display import Image as IPImage, display
display(IPImage(filename=str(dst)))
print("PASS: image generated and saved")

In [ ]:
#@title 8. STEP 8 — Automatic sharpness check (blur detection)

import numpy as np
from PIL import Image as PILImage, ImageFilter

gray = PILImage.open(dst).convert("L")
edges = gray.filter(ImageFilter.FIND_EDGES)
score = float(np.asarray(edges, dtype=np.float32).var())

if score > 60:
    verdict = "SHARP — clear, detailed"
elif score > 15:
    verdict = "CHECK — some detail, decide visually above"
else:
    verdict = "BLURRY / FLAT — do NOT proceed"

print("edge-variance sharpness score:", round(score, 1))
print("verdict:", verdict)
print()
print("Compare: a solid-color placeholder (mock backend) scores ~0-50.")
print("A clear Flux image usually scores hundreds to thousands.")

## Next steps

- **Sharp and clear above?** → Run the main notebook
  (`AnimationStudio_Colab.ipynb`) Cells 9–11 for the full Phase-1 run. The
  workflows it uses now carry the same Flux-correct settings this notebook
  verified.
- **BLURRY / no image?** → check, in order:
  1. `comfyui.log` tail: `!tail -n 40 /content/comfyui.log`
  2. Cell 3: model file size is non-zero and the symlink exists
  3. Cell 5: the model name appears in the loader list
  4. VRAM: 16 GB T4 is tight for fp8 Flux — close other notebooks/VMs
  5. The old `mock:` placeholder PNGs in `Universe/...` are *expected*; newly
     generated files overwrite them via `--persist-images`.